# Batch Candidate & Narrative Memory Extractor

### Core Architecture:
1. **Batch Ingestion**: Processes episodes in configurable batches (~19,600 raw tokens per batch, avoiding single-call output saturation).
2. **Duration-Stamped Screenplay Prompts**: Screenplay elements are pre-formatted with exact timestamps and durations:
   `[EP01_EL012 | 00:24.10->00:28.50 | dur=4.40s] **GERALD (COLDLY):** "I cannot wed a woman..."`
3. **Standardized Promo Archetypes**:
   - `immediate_shock_opener`: Visceral conflict, confrontation, or betrayal accusation within <= 3.00s.
   - `visual_anomaly_mystery_escalation`: Suspicious visual cue in CAMERA or ACTION block within <= 3.00s escalating to a twist.
   - `emotional_reversal_turning_point`: Status humiliation, vow, or power inversion within <= 3.00s building to emotional climax.
4. **Precise Trim Recommendations & Editor Directives**:
   - Verbatim opening dialogue
   - Verbatim ending dialogue
   - Internal trim quote (where internal sentence cuts occur)
   - Boundary classification (`element_boundary` vs `internal_cut`)
   - `requires_manual_editor_verification`: Explicit flag for audio/waveform verification
   - **Strict Grounding**: Timestamps are bound directly to immutable AST elements without character interpolation.
5. **Checkpoint Persistence & Resiliency**:
   - Saves `raw_llm_outputs/batch_*.json` immediately after each API call with resume capability.
   - Automated exponential backoff retry loop for transient rate limits (HTTP 429) or network timeouts.
   - Cross-references `start_element_id..end_element_id` with the AST to attach exact centisecond durations and verbatim source dialogue.


## 1. Pydantic Structured Output Schemas
Enforces strict schemas for single-episode candidates, rubric scores, trim recommendations, and narrative moments.


In [34]:
import os
import re
import json
import time
from pathlib import Path
from typing import List, Optional, Literal, Dict, Any, Tuple
from pydantic import BaseModel, Field
import pandas as pd

class TrimRecommendation(BaseModel):
    """
    Precise trim recommendations for video editors.
    Provides actionable cut instructions without unreliable character-based timestamp interpolation.
    """
    exact_opening_dialogue: str = Field(
        description="Verbatim dialogue line that begins the clip (or initial action description if visual opener)"
    )
    exact_ending_dialogue: str = Field(
        description="Verbatim dialogue line that ends the clip right before the final cliffhanger cut"
    )
    verbatim_trim_quote: Optional[str] = Field(
        default=None,
        description="Verbatim sentence/phrase where an internal cut should be made to eliminate unnecessary dialogue"
    )
    boundary_type: Literal["element_boundary", "internal_cut"] = Field(
        description="Whether the cut aligns with an element boundary or is internal to an element"
    )
    requires_manual_editor_verification: bool = Field(
        description="True if an internal cut requires human video editor verification against actual audio/video waveforms"
    )

class RubricScores(BaseModel):
    standalone_coherence: int = Field(ge=1, le=5, description="1-5 rating: conflict/stakes clear to a cold first-time viewer")
    hook_strength: int = Field(ge=1, le=5, description="1-5 rating: immediate visceral shock or intrigue within first 3 seconds")
    cliffhanger_strength: int = Field(ge=1, le=5, description="1-5 rating: cuts at peak unresolved jeopardy or open question")
    emotional_impact: int = Field(ge=1, le=5, description="1-5 rating: high-intensity betrayal, moral outrage, or dramatic reversal")

class SingleEpisodeCandidate(BaseModel):
    start_element_id: str = Field(description="First element ID of clip (e.g. EP01_EL012)")
    end_element_id: str = Field(description="Last element ID of clip (e.g. EP01_EL024)")
    hook_timestamp_offset_seconds: float = Field(
        ge=0.0, le=3.0,
        description="Offset within <= 3.0s from start of candidate where the visceral hook initiates"
    )
    archetype: Literal[
        "immediate_shock_opener",
        "visual_anomaly_mystery_escalation",
        "emotional_reversal_turning_point"
    ]
    hook_description: str = Field(description="Why the first 3 seconds compel a scrolling user to stop")
    cliffhanger_description: str = Field(description="Unresolved dramatic question or jeopardy at the final cut")
    standalone_coherence_explanation: str = Field(description="Why a cold viewer understands this conflict with 0 prior episodes")
    trim_recommendation: TrimRecommendation = Field(
        description="Precise trim recommendation detailing opening/ending dialogue, verbatim cuts, and boundary type"
    )
    rubric_scores: RubricScores

class RawNarrativeMoment(BaseModel):
    moment_id: str = Field(description="Unique ID, e.g. EP01_MOM_01")
    episode: int
    moment_type: Literal[
        "setup_or_vow",
        "clue_or_prop",
        "identity_reveal",
        "high_stakes_confrontation",
        "contrast_baseline"
    ]
    start_element_id: str
    end_element_id: str
    summary: str = Field(description="1-sentence description of the narrative beat")
    lookup_keywords: List[str] = Field(description="Key character names, props, or themes")

class CharacterAlias(BaseModel):
    canonical_name: str
    aliases: List[str]
    key_episodes: List[int]

class ObjectClue(BaseModel):
    item: str
    element_id: str
    significance: str

class Relationship(BaseModel):
    pair: str
    relation: str

class EpisodeSummary(BaseModel):
    episode: int
    summary: str

class FlatNarrativeContext(BaseModel):
    character_aliases: List[CharacterAlias] = Field(default_factory=list)
    objects_and_clues: List[ObjectClue] = Field(default_factory=list)
    relationships: List[Relationship] = Field(default_factory=list)
    episode_summaries: List[EpisodeSummary] = Field(default_factory=list)

class BatchExtractionResponse(BaseModel):
    candidates: List[SingleEpisodeCandidate] = Field(
        default_factory=list,
        description="High-converting single-episode promo candidates (target 25-55s, hard bounds 20-60s)"
    )
    narrative_moments: List[RawNarrativeMoment] = Field(
        default_factory=list,
        description="Unified cross-episode building blocks"
    )
    narrative_context: FlatNarrativeContext = Field(
        description="Flat context store: aliases, objects, relationships, and episode summaries"
    )

print("[OK] Phase 3 Pydantic Schemas loaded with TrimRecommendation support.")


[OK] Phase 3 Pydantic Schemas loaded with TrimRecommendation support.


## 2. Prompt Template & Duration-Stamped Transcript Formatter
Every screenplay element is injected into the prompt with its pre-computed duration tag `[EP01_EL012 | 00:24.10->00:28.50 | dur=4.40s]`.


In [35]:
EXTRACTION_SYSTEM_PROMPT = """You are an expert microdrama ad director and promo producer for Kuku FM.
Your task is to analyze batches of 10 microdrama episodes and extract:
1. High-Converting Single-Episode Promo Candidates (target 25.00s–55.00s, hard bounds 20.00s–60.00s)
2. Precise Trim Recommendations for Video Editors (Improvement 1.4)
3. Unified Cross-Episode Narrative Moments (tagged building blocks for cross-episode stitching)
4. Flat Narrative Context Store (aliases, clues/props, character relationships, and episode summaries)

### THREE STANDARDIZED PROMO ARCHETYPES:
1. `immediate_shock_opener`: Opens within <= 3.00s on a visceral physical action, slap, betrayal accusation, or high-stakes threat, and cuts on an unresolved cliffhanger.
2. `visual_anomaly_mystery_escalation`: Opens within <= 3.00s on a suspicious visual beat in a CAMERA or ACTION block (e.g. poison vial, royal seal, hidden scar) that rapidly escalates to a confrontation twist.
3. `emotional_reversal_turning_point`: Opens within <= 3.00s on a sharp status humiliation, vow, or power inversion that builds to an unresolved emotional climax.

### PRECISE TRIM RECOMMENDATIONS (CRITICAL EDITING INSTRUCTIONS):
For every candidate, you must return actionable trim guidance in `trim_recommendation`:
- `exact_opening_dialogue`: The exact verbatim spoken line that starts the hook (or initial visual description).
- `exact_ending_dialogue`: The exact verbatim spoken line that concludes the clip right before the cliffhanger cut.
- `verbatim_trim_quote`: If the most powerful line begins mid-element (e.g. skipping unnecessary trailing speech), provide the verbatim cut quote.
- `boundary_type`: Set to `element_boundary` if the candidate starts and ends cleanly on element borders; set to `internal_cut` if a sentence-level cut is needed.
- `requires_manual_editor_verification`: Set to True for internal cuts so video editors know to verify the edit on audio/video waveforms.
- IMPORTANT: Do NOT invent or interpolate timestamps by character counts. Character-based timestamp interpolation without actual audio waveforms is unreliable. Ground all boundaries in verbatim dialogue quotes.

### TIMING & SCORING RULES:
- Target 25.00s–55.00s total duration. Every candidate MUST strictly fall within 20.00s to 60.00s based on the pre-computed durations printed on each element!
- `hook_timestamp_offset_seconds`: MUST be between 0.0s and 3.0s. The opening hook MUST grab attention within the first 3 seconds of the clip.
- 4-Criterion Rubric (1–5 integer scores):
  - `standalone_coherence`: 5 = 100% crystal clear to a first-time viewer without prior episodes; 1-2 = confusing pronouns/backstory.
  - `hook_strength`: 5 = explosive visceral shock in 0-2s; 3 = moderate setup within 3s; 1-2 = slow opening (>3s).
  - `cliffhanger_strength`: 5 = cuts at peak jeopardy, ultimatum, or bombshell before resolution; 1-2 = conflict resolves before cut.
  - `emotional_impact`: 5 = high-intensity betrayal, moral outrage, or vindication; 1 = flat exposition.

### NARRATIVE MOMENTS (Cross-Episode Building Blocks):
Tag key narrative beats as one of:
- `setup_or_vow`: Formal promises, public oaths, or initial humiliations that demand future payoff.
- `clue_or_prop`: Tangible objects (signet rings, blood-stained clothes, letters, marriage contracts).
- `identity_reveal`: Moments where a secret status, true name, or disguised rank is disclosed.
- `high_stakes_confrontation`: Explosive clashes between key antagonists.
- `contrast_baseline`: Early suffering or weakness to contrast against later triumph.
"""

def format_batch_transcript(episodes_data: List[Dict[str, Any]]) -> str:
    """
    Formats a 10-episode batch into an element transcript with pre-computed durations:
    [EP01_EL012 | 00:24.10->00:28.50 | dur=4.40s] **GERALD (COLDLY):** "..."
    """
    transcript_blocks = []
    for ep in episodes_data:
        ep_num = ep["episode_num"]
        header = ep.get("header", {})
        logline = header.get("logline", "")
        notes = header.get("continuity_notes", "")
        block = [
            f"=== EPISODE {ep_num:02d} ===",
            f"LOGLINE: {logline}",
            f"CONTINUITY NOTES: {notes}",
            ""
        ]
        for sc in ep.get("scenes", []):
            sc_num = sc["scene_num"]
            heading = sc.get("heading", "")
            s_raw, e_raw = sc.get("start_raw", ""), sc.get("end_raw", "")
            dur_s = sc.get("duration_seconds", 0.0)
            block.append(f"--- SCENE {sc_num:02d}: {heading} [{s_raw} -> {e_raw} | dur={dur_s:.2f}s] ---")
            for el in sc.get("elements", []):
                el_id = el["element_id"]
                el_s = el["start_raw"]
                el_e = el["end_raw"]
                dur = el.get("duration_seconds", (el.get("duration_cs", 0) / 100.0))
                tag = el.get("raw_tag", el.get("element_type", "ELEMENT"))
                content = el.get("content", "").strip()
                block.append(f"[{el_id} | {el_s}->{el_e} | dur={dur:.2f}s] **{tag}:** {content}")
            block.append("")
        transcript_blocks.append("\n".join(block))
    return "\n\n".join(transcript_blocks)

print("[OK] Prompt Template and Transcript Formatter ready.")


[OK] Prompt Template and Transcript Formatter ready.


## 3. Extractor Engine with Checkpointing & AST Grounding
- **Disk Checkpointing**: Persists raw LLM outputs to `raw_llm_outputs/batch_{idx}.json` immediately after each call.
- **Exponential Backoff**: Automatically handles transient rate limits and network blips with configurable retries.
- **Sub-Batch Splitting**: Automatically splits batches into smaller slices if output capacity limits are approached.
- **AST Grounding**: Attaches verbatim AST source elements and exact centisecond durations onto narrative moments.


In [36]:
class BatchExtractorEngine:
    def __init__(
        self,
        ast_path: Optional[Path] = None,
        raw_outputs_dir: Optional[Path] = None,
        output_path: Optional[Path] = None,
        model_name: str = "gemini-3.8-flash"
    ):
        # Defensive path resolution: works both from repo root and within candidate_extractor/
        if ast_path:
            self.ast_path = Path(ast_path)
        elif Path("ast_builder/01_structured_episodes.json").exists():
            self.ast_path = Path("ast_builder/01_structured_episodes.json")
        elif Path("../ast_builder/01_structured_episodes.json").exists():
            self.ast_path = Path("../ast_builder/01_structured_episodes.json")
        elif Path("01_structured_episodes.json").exists():
            self.ast_path = Path("01_structured_episodes.json")
        else:
            self.ast_path = Path("ast_builder/01_structured_episodes.json")

        if raw_outputs_dir:
            self.raw_outputs_dir = Path(raw_outputs_dir)
        elif Path("candidate_extractor/raw_llm_outputs").exists():
            self.raw_outputs_dir = Path("candidate_extractor/raw_llm_outputs")
        elif Path("raw_llm_outputs").exists():
            self.raw_outputs_dir = Path("raw_llm_outputs")
        elif Path("candidate_extractor").exists():
            self.raw_outputs_dir = Path("candidate_extractor/raw_llm_outputs")
        else:
            self.raw_outputs_dir = Path("raw_llm_outputs")

        if output_path:
            self.output_path = Path(output_path)
        elif Path("candidate_extractor").exists():
            self.output_path = Path("candidate_extractor/02_batch_extractions.json")
        else:
            self.output_path = Path("02_batch_extractions.json")

        self.model_name = model_name
        self.raw_outputs_dir.mkdir(parents=True, exist_ok=True)
        
        if not self.ast_path.exists():
            raise FileNotFoundError(f"AST not found at {self.ast_path}. Run AST builder first!")
        self.ast_data = json.loads(self.ast_path.read_text(encoding="utf-8"))
        
        # Build quick element lookup table
        self.element_lookup: Dict[str, Dict[str, Any]] = {}
        for ep in self.ast_data.get("episodes", []):
            for sc in ep.get("scenes", []):
                for el in sc.get("elements", []):
                    self.element_lookup[el["element_id"]] = el

        self.client = self._init_client()

    def _init_client(self):
        api_key = os.environ.get("GEMINI_API_KEY")
        if not api_key:
            print("[WARN] GEMINI_API_KEY environment variable not detected.")
        try:
            from google import genai
            return genai.Client(api_key=api_key) if api_key else genai.Client()
        except Exception as e:
            print(f"[WARN] Could not initialize google-genai Client: {e}")
            return None

    def call_llm_batch(self, batch_index: int, episodes_slice: List[Dict[str, Any]]) -> BatchExtractionResponse:
        """Calls Gemini with structured Pydantic schema enforcing precise trim recommendations."""
        from google.genai import types
        
        transcript = format_batch_transcript(episodes_slice)
        batch_prompt = (
            f"{EXTRACTION_SYSTEM_PROMPT}\n\n"
            f"Here is the complete screenplay transcript for Batch {batch_index} "
            f"(Episodes {episodes_slice[0]['episode_num']} to {episodes_slice[-1]['episode_num']}):\n\n"
            f"{transcript}\n\n"
            f"Extract all single-episode promo candidates (targeting 25-55s duration), "
            f"precise trim recommendations, cross-episode narrative moments, and flat narrative context according to the JSON schema."
        )

        config = types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=BatchExtractionResponse,
            temperature=0.2,
        )

        response = self.client.models.generate_content(
            model=self.model_name,
            contents=batch_prompt,
            config=config
        )

        if hasattr(response, "parsed") and response.parsed:
            return response.parsed
        return BatchExtractionResponse.model_validate_json(response.text)

    def process_batch(
        self, 
        batch_index: Any, 
        episodes_slice: List[Dict[str, Any]], 
        force_refresh: bool = False,
        max_retries: int = 3,
        retry_delay: float = 5.0
    ) -> Dict[str, Any]:
        """Processes a single batch with disk checkpointing and exponential backoff retries."""
        checkpoint_file = self.raw_outputs_dir / f"batch_{batch_index}.json"
        
        if checkpoint_file.exists() and not force_refresh:
            print(f"[CHECKPOINT] Batch {batch_index} found at {checkpoint_file.name}. Loading from disk.")
            return json.loads(checkpoint_file.read_text(encoding="utf-8"))

        start_ep = episodes_slice[0]["episode_num"]
        end_ep = episodes_slice[-1]["episode_num"]
        print(f"[API CALL] Processing Batch {batch_index} (Episodes {start_ep}..{end_ep}) using {self.model_name}...")
        
        for attempt in range(1, max_retries + 1):
            try:
                t0 = time.time()
                result_obj = self.call_llm_batch(batch_index, episodes_slice)
                elapsed = time.time() - t0
                result_dict = result_obj.model_dump()
                print(f"           Batch {batch_index} completed in {elapsed:.2f}s "
                      f"({len(result_dict['candidates'])} candidates, {len(result_dict['narrative_moments'])} moments)")
                
                # Atomic checkpoint persistence to disk
                checkpoint_file.write_text(json.dumps(result_dict, indent=2), encoding="utf-8")
                print(f"           Saved checkpoint to {checkpoint_file.name}")
                return result_dict

            except Exception as e:
                print(f"[WARN] Batch {batch_index} attempt {attempt}/{max_retries} failed: {e}")
                if attempt < max_retries:
                    backoff = retry_delay * (2 ** (attempt - 1))
                    print(f"       Waiting {backoff:.1f}s before retrying Batch {batch_index}...")
                    time.sleep(backoff)
                else:
                    if len(episodes_slice) > 5:
                        print(f"[SPLIT] Splitting Batch {batch_index} into two 5-episode sub-batches...")
                        half = len(episodes_slice) // 2
                        res1 = self.process_batch(f"{batch_index}a", episodes_slice[:half], force_refresh=force_refresh, max_retries=max_retries, retry_delay=retry_delay)
                        res2 = self.process_batch(f"{batch_index}b", episodes_slice[half:], force_refresh=force_refresh, max_retries=max_retries, retry_delay=retry_delay)
                        merged = {
                            "candidates": res1["candidates"] + res2["candidates"],
                            "narrative_moments": res1["narrative_moments"] + res2["narrative_moments"],
                            "narrative_context": {
                                "character_aliases": res1["narrative_context"].get("character_aliases", []) + res2["narrative_context"].get("character_aliases", []),
                                "objects_and_clues": res1["narrative_context"].get("objects_and_clues", []) + res2["narrative_context"].get("objects_and_clues", []),
                                "relationships": res1["narrative_context"].get("relationships", []) + res2["narrative_context"].get("relationships", []),
                                "episode_summaries": res1["narrative_context"].get("episode_summaries", []) + res2["narrative_context"].get("episode_summaries", []),
                            }
                        }
                        checkpoint_file.write_text(json.dumps(merged, indent=2), encoding="utf-8")
                        return merged
                    else:
                        print(f"[ERROR] Batch {batch_index} exhausted all {max_retries} attempts.")
                        raise

    def enrich_narrative_moment_with_ast(self, moment: Dict[str, Any]) -> Dict[str, Any]:
        """AST Grounding: Looks up start_element_id..end_element_id in AST and attaches verbatim elements."""
        s_id = moment.get("start_element_id")
        e_id = moment.get("end_element_id")
        s_el = self.element_lookup.get(s_id)
        e_el = self.element_lookup.get(e_id)
        
        if not s_el or not e_el:
            moment["duration_seconds"] = 0.0
            moment["source_elements_verbatim"] = []
            return moment
            
        ep_num = s_el["episode_num"]
        s_num = int(re.search(r"EL(\d+)", s_id).group(1))
        e_num = int(re.search(r"EL(\d+)", e_id).group(1))
        
        duration_cs = e_el["end_cs"] - s_el["start_cs"]
        moment["duration_seconds"] = round(duration_cs / 100.0, 2)
        moment["duration_cs"] = duration_cs
        
        verbatim_lines = []
        for el_num in range(s_num, e_num + 1):
            curr_id = f"EP{ep_num:02d}_EL{el_num:03d}"
            curr_el = self.element_lookup.get(curr_id)
            if curr_el:
                tag = curr_el.get("raw_tag", curr_el.get("element_type", "ELEMENT"))
                content = curr_el.get("content", "").strip()
                s_raw, e_raw = curr_el.get("start_raw", ""), curr_el.get("end_raw", "")
                verbatim_lines.append(f"[{curr_id} | {s_raw}-{e_raw}] **{tag}:** {content}")
                
        moment["source_elements_verbatim"] = verbatim_lines
        return moment

    def run_all(
        self, 
        batch_size: int = 5, 
        force_refresh: bool = False,
        max_retries: int = 3,
        retry_delay: float = 5.0
    ) -> Dict[str, Any]:
        """Executes extraction across all 50 episodes in 5 batches with AST enrichment."""
        episodes = self.ast_data.get("episodes", [])
        total_episodes = len(episodes)
        total_batches = (total_episodes + batch_size - 1) // batch_size
        
        print("=" * 70)
        print(f"STARTING BATCH EXTRACTION: {total_episodes} Episodes into {total_batches} Batches")
        print("=" * 70)
        
        all_candidates = []
        all_narrative_moments = []
        merged_aliases = []
        merged_clues = []
        merged_relationships = []
        merged_summaries = []
        
        for b_idx in range(1, total_batches + 1):
            start_i = (b_idx - 1) * batch_size
            end_i = min(start_i + batch_size, total_episodes)
            batch_slice = episodes[start_i:end_i]
            
            batch_result = self.process_batch(b_idx, batch_slice, force_refresh=force_refresh, max_retries=max_retries, retry_delay=retry_delay)
            
            for cand in batch_result.get("candidates", []):
                s_el = self.element_lookup.get(cand["start_element_id"])
                e_el = self.element_lookup.get(cand["end_element_id"])
                if s_el and e_el:
                    dur_cs = e_el["end_cs"] - s_el["start_cs"]
                    cand["calculated_duration_seconds"] = round(dur_cs / 100.0, 2)
                    cand["calculated_duration_cs"] = dur_cs
                all_candidates.append(cand)
                
            for mom in batch_result.get("narrative_moments", []):
                enriched_mom = self.enrich_narrative_moment_with_ast(mom)
                all_narrative_moments.append(enriched_mom)
                
            ctx = batch_result.get("narrative_context", {})
            merged_aliases.extend(ctx.get("character_aliases", []))
            merged_clues.extend(ctx.get("objects_and_clues", []))
            merged_relationships.extend(ctx.get("relationships", []))
            merged_summaries.extend(ctx.get("episode_summaries", []))

        final_payload = {
            "series_title": self.ast_data.get("corpus_title", "Microdrama Series"),
            "total_episodes": total_episodes,
            "total_single_episode_candidates": len(all_candidates),
            "total_narrative_moments": len(all_narrative_moments),
            "candidates": all_candidates,
            "narrative_moments": all_narrative_moments,
            "narrative_context": {
                "character_aliases": merged_aliases,
                "objects_and_clues": merged_clues,
                "relationships": merged_relationships,
                "episode_summaries": merged_summaries
            }
        }
        
        self.output_path.write_text(json.dumps(final_payload, indent=2), encoding="utf-8")
        file_size_kb = self.output_path.stat().st_size / 1024
        
        print("\n" + "=" * 70)
        print("CANDIDATE & NARRATIVE MOMENT EXTRACTION COMPLETE")
        print("=" * 70)
        print(f"Single-Episode Candidates Extracted: {len(all_candidates):4d}")
        print(f"Narrative Moments (AST Attached)   : {len(all_narrative_moments):4d}")
        print(f"Character Aliases Mapped           : {len(merged_aliases):4d}")
        print(f"Key Props and Clues Cataloged      : {len(merged_clues):4d}")
        print(f"Saved merged extractions to        : {self.output_path.resolve()} ({file_size_kb:.1f} KB)")
        print("=" * 70)
        
        return final_payload

print("[OK] BatchExtractorEngine compiled successfully.")


[OK] BatchExtractorEngine compiled successfully.


In [37]:
import os
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

# 1. Safely resolve the absolute path to models.env as a string
env_path = str(Path(r"C:\Users\Anirudh Singh\OneDrive\Documents\Kuku FM") / "models.env")

# 2. Load it, forcing it to override any empty cached values
load_dotenv(dotenv_path=env_path, override=True)

api_key = os.getenv("GEMINI_API_KEY")

if api_key:
    masked = api_key[:4] + "..." + api_key[-4:] if len(api_key) > 8 else "***"
    print(f"[OK] GEMINI_API_KEY is detected: {masked}")
    
    # Initialize client
    client = genai.Client(api_key=api_key)
else:
    print("[WARN] GEMINI_API_KEY is not currently set in environment variables.")
    print(f"-> Looked for file at: {env_path}")

[OK] GEMINI_API_KEY is detected: AQ.A...ZORQ


## 4. Pre-Flight Verification & Prompt Preview
Initializes the engine, checks environment keys, and displays a preview of the formatted transcript for Batch 1.


In [38]:
extractor = BatchExtractorEngine(
    model_name="gemini-3.8-flash"
)

# Preview formatted batch transcript for Episodes 1-10
batch_1_episodes = extractor.ast_data["episodes"][:10]
sample_transcript = format_batch_transcript(batch_1_episodes)
print("=" * 70)
print(f"BATCH 1 PROMPT TRANSCRIPT PREVIEW ({len(sample_transcript.splitlines())} lines):")
print("=" * 70)
for line in sample_transcript.splitlines()[:15]:
    print(line)
print("... [Truncated for preview] ...")


BATCH 1 PROMPT TRANSCRIPT PREVIEW (926 lines):
=== EPISODE 01 ===
LOGLINE: Lady Juliana arrives at the cathedral in her late mother's bridal gown, only to suffer the ultimate betrayal when her fiancé Gerald abandons her at the altar for her cruel sister Clarissa with her family's full blessing.
CONTINUITY NOTES: First episode of the series. Establishes Juliana's humiliation and abandonment by her betrothed, Gerald, and her noble family in favor of her sister, Clarissa.

--- SCENE 01: INT. CATHEDRAL – DAY [00:00.00 -> 02:18.00 | dur=138.00s] ---
[EP01_EL001 | 00:00.00->00:04.80 | dur=4.80s] **CAMERA:** Symmetrical framing on the heavy church doors as they slowly part, pushing in slightly as Juliana stands framed in the light, dressed in an elegant white silk gown.
[EP01_EL002 | 00:01.00->00:04.80 | dur=3.80s] **ACTION:** Juliana pauses at the threshold, her expression bright with joyful anticipation, wearing her mother's vintage gown and an oval pendant necklace.
[EP01_EL003 | 00:04.80-

## 5. Execute Batch Extraction Across All Episodes
Runs batch extraction with automatic retries and immediate disk checkpointing.


In [39]:
import os
import pandas as pd

# Check checkpoints safely
# raw_dir = getattr(extractor, "raw_outputs_dir", None)
# has_checkpoints = (
#     any(raw_dir.glob("batch_*.json"))
#     if raw_dir and raw_dir.is_dir()
#     else False
# )

# Verify either client/key exists or we can use existing checkpoints
has_api_access = bool(os.getenv("GEMINI_API_KEY") or (client is not None))

if not has_api_access and not has_checkpoints:
    print("=" * 70)
    print("[ACTION REQUIRED] GEMINI_API_KEY is needed to call Gemini:")
    print("  In PowerShell: $env:GEMINI_API_KEY = 'your_key_here'")
    print("  Or run Cell 1 with a valid models.env file.")
    print("=" * 70)
    extraction_results = {"candidates": [], "narrative_moments": []}
else:
    # If your extractor object needs the client explicitly:
    # extractor.client = client

    # Run extraction
    extraction_results = extractor.run_all(batch_size=5, force_refresh=False, max_retries=3, retry_delay=5.0)
    
    candidates = extraction_results.get("candidates") or []

    if candidates:
        rows = []
        for c in candidates:
            trim = c.get("trim_recommendation") or {}
            rubric = c.get("rubric_scores") or {}

            hook_offset = c.get("hook_timestamp_offset_seconds")
            hook_str = f"{hook_offset:.2f}s" if hook_offset is not None else "N/A"

            rows.append({
                "Start": c.get("start_element_id"),
                "End": c.get("end_element_id"),
                "Dur (s)": c.get("calculated_duration_seconds", 0.0),
                "Hook Offset": hook_str,
                "Archetype": c.get("archetype", "Unknown"),
                "Boundary Type": trim.get("boundary_type", "N/A"),
                "Needs Editor Check": trim.get("requires_manual_editor_verification", False),
                "Score": sum(rubric.values()) if rubric else 0.0
            })

        df_cands = pd.DataFrame(rows)
        print("\n--- EXTRACTED CANDIDATES WITH PRECISE TRIM RECOMMENDATIONS ---")
        print(df_cands.head(10).to_string(index=False))
    else:
        print("\nNo candidates found in extraction results.")

STARTING BATCH EXTRACTION: 50 Episodes into 10 Batches
[CHECKPOINT] Batch 1 found at batch_1.json. Loading from disk.
[CHECKPOINT] Batch 2 found at batch_2.json. Loading from disk.
[CHECKPOINT] Batch 3 found at batch_3.json. Loading from disk.
[CHECKPOINT] Batch 4 found at batch_4.json. Loading from disk.
[CHECKPOINT] Batch 5 found at batch_5.json. Loading from disk.
[CHECKPOINT] Batch 6 found at batch_6.json. Loading from disk.
[CHECKPOINT] Batch 7 found at batch_7.json. Loading from disk.
[CHECKPOINT] Batch 8 found at batch_8.json. Loading from disk.
[CHECKPOINT] Batch 9 found at batch_9.json. Loading from disk.
[CHECKPOINT] Batch 10 found at batch_10.json. Loading from disk.

CANDIDATE & NARRATIVE MOMENT EXTRACTION COMPLETE
Single-Episode Candidates Extracted:   42
Narrative Moments (AST Attached)   :   84
Character Aliases Mapped           :   58
Key Props and Clues Cataloged      :   34
Saved merged extractions to        : C:\Users\Anirudh Singh\OneDrive\Documents\Kuku FM\candidat

## 6. Deep Inspection of Precise Trim Recommendations & AST Attachments
Inspects the exact dialogue boundaries, trim quotes, and AST verbatim elements for video editors.


In [40]:
cands = extraction_results.get("candidates", [])
if cands:
    sample_cand = cands[0]
    trim = sample_cand["trim_recommendation"]
    print("=" * 75)
    print(f"SAMPLE PROMO CANDIDATE: {sample_cand['start_element_id']} -> {sample_cand['end_element_id']}")
    print(f"Archetype             : {sample_cand['archetype']}")
    print(f"AST Duration          : {sample_cand.get('calculated_duration_seconds', 0.0):.2f}s")
    print(f"Hook Offset           : {sample_cand['hook_timestamp_offset_seconds']:.2f}s")
    print("-" * 75)
    print("PRECISE TRIM RECOMMENDATIONS (FOR VIDEO EDITORS):")
    print(f"  Exact Opening Dialogue: {trim['exact_opening_dialogue']}")
    print(f"  Exact Ending Dialogue : {trim['exact_ending_dialogue']}")
    print(f"  Verbatim Trim Quote   : {trim['verbatim_trim_quote']}")
    print(f"  Boundary Type         : {trim['boundary_type']}")
    print(f"  Needs Editor Check    : {trim['requires_manual_editor_verification']}")
    print("-" * 75)
    print(f"Hook Description      : {sample_cand['hook_description']}")
    print(f"Cliffhanger Reason    : {sample_cand['cliffhanger_description']}")
    print(f"Rubric Scores         : {sample_cand['rubric_scores']}")
else:
    print("[INFO] Run extraction with GEMINI_API_KEY to inspect candidates.")


SAMPLE PROMO CANDIDATE: EP01_EL063 -> EP01_EL092
Archetype             : emotional_reversal_turning_point
AST Duration          : 46.20s
Hook Offset           : 0.10s
---------------------------------------------------------------------------
PRECISE TRIM RECOMMENDATIONS (FOR VIDEO EDITORS):
  Exact Opening Dialogue: Enough! You have embarrassed this family long enough.
  Exact Ending Dialogue : ...when Clarissa is the true jewel of the county.
  Verbatim Trim Quote   : None
  Boundary Type         : element_boundary
  Needs Editor Check    : False
---------------------------------------------------------------------------
Hook Description      : Matriarch roars 'Enough! You have embarrassed this family long enough', initiating an immediate public humiliation before a crowded cathedral.
Cliffhanger Reason    : Gerald publicly brands Juliana worthless and declares Clarissa the true jewel as Juliana's expression hardens into cold defiance before the final cut.
Rubric Scores         : {'s